# Security testing and fuzzing an MCP boundary

This notebook turns tenant, protocol, schema, approval, and side-effect invariants into a bounded executable campaign. It compares a deliberately vulnerable baseline with a hardened target and judges both using an independent oracle plus observable state. No network or production system is contacted.

## Learning objectives

You will inspect a real official-SDK exchange, generate structure-aware mutations, detect a completed forbidden disclosure, shrink and preserve the reproducer, test metamorphic relations and approval replay, and calculate metrics with explicit populations.

In [ ]:
import json
import runpy
from dataclasses import replace
from pathlib import Path

lab = runpy.run_path(Path('lab.py'))
now = lab['utc']('2026-09-29T04:00:00Z')
Decision = lab['Decision']
CaseClass = lab['CaseClass']


## 1. Probe the real SDK boundary

The official Python SDK runs client and server in memory. The probe checks discovery, a valid call, a domain denial, and an extra argument. Reviewed middleware closes the pinned SDK line's permissive top-level extra-property behavior before the handler runs.

In [ ]:
sdk = await lab['run_sdk_probe']()
sdk


## 2. Specify expected behavior independently

A case fixes raw bytes, routing headers, trusted identity, an expected decision/reason, and an expected effect delta. The target does not supply its own label.

In [ ]:
cases = lab['generated_cases'](seed=1401, max_cases=10)
cross_tenant = next(case for case in cases if case.case_id == 'cross-tenant-read')
{
    'case_id': cross_tenant.case_id,
    'class': cross_tenant.case_class,
    'mutation': cross_tenant.mutation,
    'expected': cross_tenant.expected_decision,
    'expected_reason': cross_tenant.expected_reason,
    'expected_effect_delta': cross_tenant.expected_effect_delta,
    'input_digest': lab['digest'](cross_tenant.raw),
}


## 3. Run the vulnerable baseline

The baseline intentionally omits tenant ownership enforcement. A cross-tenant request returns another tenant's summary and creates a data-release event. The oracle checks the separate ledger, so a misleading response string could not hide that disclosure.

In [ ]:
vulnerable = lab['run_campaign'](lab['VulnerableMcpGateway'], cases, now)
vulnerable.metrics()


In [ ]:
failure = next(result for result in vulnerable.results if not result.oracle.passed)
{
    'case': failure.case.case_id,
    'actual_decision': failure.observation.decision,
    'effect_delta': failure.effect_delta,
    'violations': failure.oracle.violations,
}


## 4. Run the hardened target

The same corpus now exercises strict parsing, header/body agreement, exact tool arguments, trusted scope, tenant ownership, and uniform denial. Reusing the same cases makes the architectural comparison explicit.

In [ ]:
hardened = lab['run_campaign'](lab['McpGateway'], cases, now)
assert hardened.oracle_passes == hardened.total_cases
hardened.metrics()


## 5. Inspect mutation coverage

The small deterministic catalogue separates valid, attack, and robustness populations. Production campaigns add schema-derived, stateful, coverage-guided, and fault-injection cases while preserving these labels.

In [ ]:
[
    {
        'id': result.case.case_id,
        'class': result.case.case_class,
        'mutation': result.case.mutation,
        'decision': result.observation.decision,
        'reason': result.observation.reason_code,
        'oracle_passed': result.oracle.passed,
    }
    for result in hardened.results
]


## 6. Test a metamorphic relation

Object key ordering and insignificant JSON whitespace should not change an authorized request's semantics. This relation is useful because it defines expected behavior without enumerating every encoding.

In [ ]:
identity = lab['TrustedIdentity']('analyst-42', 'acme', frozenset({'ticket:read'}))
raw = lab['modern_request']('ticket.read', {'ticket_id': 'acme-7'})
pretty = json.dumps(json.loads(raw), indent=4).encode()
first = lab['McpGateway']().handle(raw, lab['headers'](), identity, now)
second = lab['McpGateway']().handle(pretty, lab['headers'](), identity, now)
assert first.decision == second.decision == Decision.ALLOW
{'canonical_bytes': len(raw), 'pretty_bytes': len(pretty), 'decision': first.decision}


## 7. Shrink and preserve the failure

The lab's small delta debugger removes optional capability metadata only if the forbidden effect still occurs. Mature property and coverage-guided tools perform deeper shrinking or corpus minimization.

In [ ]:
shrunk = lab['shrink_failure'](
    failure.case,
    lambda candidate: lab['run_case'](
        lab['VulnerableMcpGateway'](), candidate, now
    ).effect_delta > 0,
)
{
    'before_bytes': len(failure.case.raw),
    'after_bytes': len(shrunk.raw),
    'still_forbidden': lab['run_case'](
        lab['VulnerableMcpGateway'](), shrunk, now
    ).effect_delta == 1,
}


In [ ]:
lab['failure_artifact'](failure)


## 8. Test a stateful approval sequence

An exact reply plan receives one trusted receipt. The first call executes; replay is denied and the ledger still contains one effect. Changing body, principal, target, or operation ID similarly invalidates the binding.

In [ ]:
target = lab['McpGateway']()
approved, receipt = lab['approved_reply_fixture'](target, now)
first = lab['run_case'](target, approved, now)
replay_case = replace(
    approved,
    case_id='approval-replay',
    case_class=CaseClass.ATTACK,
    expected_decision=Decision.DENY,
    expected_reason='APPROVAL_REPLAYED',
    expected_effect_delta=0,
)
replay = lab['run_case'](target, replay_case, now)
{
    'first': first.observation.reason_code,
    'replay': replay.observation.reason_code,
    'effects': target.ledger.count(),
}


## 9. Interpret metrics with denominators

The baseline has four labelled attack attempts and one completed forbidden disclosure, so attack success is 1/4. The two valid attempts are a separate false-block population, and four malformed/boundary cases form the robustness population.

In [ ]:
{
    'attack_population': vulnerable.attack_attempts,
    'completed_forbidden_effects': vulnerable.completed_forbidden_effects,
    'valid_population': vulnerable.valid_attempts,
    'robustness_population': vulnerable.robustness_cases,
    **vulnerable.metrics(),
}


## Production mapping

Use Hypothesis for typed and stateful properties, Schemathesis for an HTTP/OpenAPI management surface, Atheris or native engines plus sanitizers for parser paths, and OSS-Fuzz or ClusterFuzzLite for continuous campaigns. Run disposable targets with synthetic data, hard resource and network limits, authenticated state observation, persistent restricted artifacts, deduplication, accountable triage, and a deterministic regression tier in every PR.

## Reflection and exercises

1. Why can a target return `deny` and still fail the security oracle?
2. Which parts of this corpus are attacks, robustness cases, and valid controls?
3. Add `resources/read` and define URI-encoding metamorphic relations.
4. Build a Hypothesis state machine for issue, mutate, expire, consume, and replay.
5. Wrap `parse_request` in Atheris. Which seeds and protocol dictionary tokens reach deep paths fastest?
6. Design a reset oracle that proves a fuzz case left no cache, queue, database, or background-task residue.